# PriceIQ Agent — Phase 1 Prototype
**JHU Carey | Generative AI | Team: Kangchun Sun, Tao Cheng, Maoyuan Li**

This notebook implements the **Planner + Executor** dual-agent architecture using the Claude Agent SDK.  
It demonstrates the core PriceIQ workflow: price elasticity analysis + demand signals → revenue simulation.

Use claude api key inside.


## 0. Install Dependencies

In [13]:
# Install required packages
!pip install anthropic pytrends scipy pandas numpy --quiet
print("✅ Packages installed")


✅ Packages installed


## 1. Imports & API Key

In [14]:
import os, json, sqlite3, warnings
import pandas as pd
import numpy as np
from scipy import stats
import anthropic
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# ── Secure API key retrieval (Colab Secrets → env var fallback) ──────────────
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
    if ANTHROPIC_API_KEY:
        print("✅ API key loaded from environment variable")
    else:
        print("⚠️  No API key found. Set ANTHROPIC_API_KEY in Colab Secrets.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"Anthropic SDK version: {anthropic.__version__}")


✅ API key loaded from Colab Secrets
Anthropic SDK version: 0.96.0


## 2. Data Setup — Synthetic Olist Database

We generate a synthetic database that mirrors the Olist Brazilian E-Commerce schema.  
*(Optional: replace with real Olist SQLite download from Kaggle for production.)*


In [15]:
def build_synthetic_olist_db() -> sqlite3.Connection:
    """
    Generate synthetic order data mimicking Olist schema.
    Real Olist has 100K orders (2016-2018); this prototype uses 2,000 rows
    per category so regressions have statistical power.
    """
    conn = sqlite3.connect(':memory:')
    np.random.seed(42)

    CATEGORIES = {
        'esporte_lazer':         {'mean_price': 45,  'vol_sensitivity': -1.4, 'base_vol': 600},
        'eletronicos':           {'mean_price': 120, 'vol_sensitivity': -0.7, 'base_vol': 300},
        'moda_bolsas':           {'mean_price': 80,  'vol_sensitivity': -1.1, 'base_vol': 450},
        'casa_mesa_banho':       {'mean_price': 60,  'vol_sensitivity': -0.9, 'base_vol': 400},
        'informatica_acessorios':{'mean_price': 95,  'vol_sensitivity': -0.8, 'base_vol': 350},
    }

    rows = []
    for cat, params in CATEGORIES.items():
        # Generate price points: 5 buckets around mean
        prices = np.random.lognormal(np.log(params['mean_price']), 0.4, 2000)
        for i, price in enumerate(prices):
            # Volume inversely related to price via elasticity
            noise = np.random.normal(0, 0.15)
            log_vol = np.log(params['base_vol']) + params['vol_sensitivity'] * np.log(price / params['mean_price']) + noise
            qty = max(1, int(np.exp(log_vol)))
            date = datetime(2017, 1, 1) + timedelta(days=np.random.randint(0, 580))
            rows.append({
                'product_category_name': cat,
                'price': round(float(price), 2),
                'order_qty': qty,
                'order_date': date.strftime('%Y-%m-%d'),
                'freight_value': round(float(price) * np.random.uniform(0.05, 0.15), 2),
            })

    df = pd.DataFrame(rows)
    df.to_sql('order_items', conn, index=False, if_exists='replace')

    # Create a price_buckets view for efficient aggregation
    conn.execute("""
        CREATE VIEW IF NOT EXISTS price_volume_buckets AS
        SELECT
            product_category_name,
            ROUND(price / 10) * 10 AS price_bucket,
            COUNT(*) AS order_count,
            AVG(price) AS avg_price,
            SUM(order_qty) AS total_qty,
            AVG(freight_value) AS avg_freight,
            strftime('%Y-%m', order_date) AS year_month
        FROM order_items
        GROUP BY product_category_name, price_bucket, year_month
    """)
    conn.commit()

    print(f"✅ Synthetic Olist DB created: {len(df):,} orders, {df['product_category_name'].nunique()} categories")
    print(f"   Categories: {', '.join(df['product_category_name'].unique())}")
    return conn

DB_CONN = build_synthetic_olist_db()


✅ Synthetic Olist DB created: 10,000 orders, 5 categories
   Categories: esporte_lazer, eletronicos, moda_bolsas, casa_mesa_banho, informatica_acessorios


## 3. Tool Implementations (4 Tools)

Each tool is a pure Python function. All analytical transformations use pandas/scipy — no LLM involved.


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# TOOL 1: query_sales_data
# ══════════════════════════════════════════════════════════════════════════════
def query_sales_data(category: str, start_date: str = '2017-01', end_date: str = '2018-08') -> dict:
    """
    Query Olist SQLite for price-volume distribution by category and date range.
    Returns a dict with aggregated price-volume data.
    """
    try:
        query = """
            SELECT
                price_bucket,
                AVG(avg_price) AS avg_price,
                SUM(total_qty) AS total_qty,
                COUNT(*) AS months_active,
                AVG(avg_freight) AS avg_freight
            FROM price_volume_buckets
            WHERE product_category_name = ?
              AND year_month BETWEEN ? AND ?
            GROUP BY price_bucket
            ORDER BY price_bucket
        """
        df = pd.read_sql_query(query, DB_CONN, params=(category, start_date, end_date))

        if df.empty:
            return {"status": "error", "message": f"No data for category '{category}' in range {start_date}–{end_date}"}

        # Filter out extreme outliers (top/bottom 5% price buckets)
        q_low, q_high = df['avg_price'].quantile(0.05), df['avg_price'].quantile(0.95)
        df = df[(df['avg_price'] >= q_low) & (df['avg_price'] <= q_high)]

        return {
            "status": "ok",
            "category": category,
            "n_price_buckets": len(df),
            "price_range": {"min": round(df['avg_price'].min(), 2), "max": round(df['avg_price'].max(), 2)},
            "total_orders": int(df['total_qty'].sum()),
            "avg_freight": round(df['avg_freight'].mean(), 2),
            "data": df.to_dict(orient='records')
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 2: calculate_price_elasticity
# ══════════════════════════════════════════════════════════════════════════════
def calculate_price_elasticity(category: str, start_date: str = '2017-01', end_date: str = '2018-08') -> dict:
    """
    Compute log-linear price elasticity via OLS regression: ln(Q) = α + β·ln(P)
    β is the price elasticity coefficient. β < -1 = elastic (price-sensitive).
    """
    try:
        sales_result = query_sales_data(category, start_date, end_date)
        if sales_result['status'] == 'error':
            return sales_result

        df = pd.DataFrame(sales_result['data'])

        # Require at least 5 data points for a valid regression
        if len(df) < 5:
            return {"status": "error", "message": "Insufficient data points for regression (need ≥5)"}

        # Log-linear regression: ln(Q) = α + β·ln(P)
        log_price = np.log(df['avg_price'].values)
        log_qty   = np.log(df['total_qty'].values + 1)  # +1 to avoid log(0)

        slope, intercept, r_value, p_value, std_err = stats.linregress(log_price, log_qty)

        # 95% confidence interval for β
        t_crit = stats.t.ppf(0.975, df=len(df) - 2)
        ci_low  = round(slope - t_crit * std_err, 3)
        ci_high = round(slope + t_crit * std_err, 3)

        elasticity_label = "elastic" if slope < -1 else "inelastic" if slope > -1 else "unit-elastic"

        return {
            "status": "ok",
            "category": category,
            "elasticity_beta": round(slope, 3),
            "r_squared": round(r_value ** 2, 3),
            "p_value": round(p_value, 4),
            "std_err": round(std_err, 3),
            "confidence_interval_95": [ci_low, ci_high],
            "n_observations": len(df),
            "interpretation": elasticity_label,
            "rule_of_thumb": f"A 10% price change → ~{abs(slope * 10):.1f}% volume change"
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 3: get_demand_signals
# ══════════════════════════════════════════════════════════════════════════════
# Brazilian holiday calendar (static — no API key needed)
BRAZIL_HOLIDAYS = [
    ("2024-01-01", "New Year"),    ("2024-02-12", "Carnival"),
    ("2024-04-21", "Tiradentes"), ("2024-05-01", "Labor Day"),
    ("2024-06-20", "Corpus Christi"), ("2024-09-07", "Independence Day"),
    ("2024-10-12", "Our Lady"), ("2024-11-02", "All Souls"),
    ("2024-11-15", "Republic Day"), ("2024-11-29", "Black Friday"),
    ("2024-12-25", "Christmas"),  ("2025-01-01", "New Year"),
    ("2025-02-03", "Carnival"),   ("2025-11-28", "Black Friday"),
    ("2025-12-25", "Christmas"),  ("2026-11-27", "Black Friday"),
    ("2026-12-25", "Christmas"),
]

def get_demand_signals(category: str, country: str = 'BR') -> dict:
    """
    Fetch real-time demand signals:
    1. Google Trends search index (pytrends) — graceful fallback to seasonal estimate
    2. Days to next major Brazilian holiday → demand multiplier
    """
    # ── Step 1: Google Trends ────────────────────────────────────────────────
    trend_index = None
    trend_source = "live"
    category_keywords = {
        'esporte_lazer':          'equipamentos esportivos',
        'eletronicos':            'eletrônicos',
        'moda_bolsas':            'bolsas femininas',
        'casa_mesa_banho':        'decoração casa',
        'informatica_acessorios': 'acessórios computador',
    }
    keyword = category_keywords.get(category, category.replace('_', ' '))

    try:
        from pytrends.request import TrendReq
        pytrends = TrendReq(hl='pt-BR', tz=-180, timeout=(5, 15))
        pytrends.build_payload([keyword], cat=0, timeframe='today 3-m', geo=country)
        df_trends = pytrends.interest_over_time()
        if not df_trends.empty and keyword in df_trends.columns:
            recent = df_trends[keyword].tail(4).mean()  # last 4 weeks
            baseline = df_trends[keyword].mean()
            trend_index = round(float(recent / baseline) if baseline > 0 else 1.0, 3)
        else:
            raise ValueError("Empty trends response")
    except Exception as e:
        # GRACEFUL DEGRADATION: use seasonal baseline (month-based estimates)
        month = datetime.now().month
        seasonal_map = {
            1: 0.85, 2: 0.90, 3: 0.95, 4: 1.00, 5: 1.05, 6: 1.00,
            7: 1.05, 8: 1.05, 9: 1.10, 10: 1.15, 11: 1.30, 12: 1.25
        }
        trend_index = seasonal_map.get(month, 1.0)
        trend_source = f"seasonal_estimate (pytrends unavailable: {str(e)[:60]})"

    # ── Step 2: Holiday Proximity ────────────────────────────────────────────
    today = datetime.now().date()
    upcoming = []
    for date_str, name in BRAZIL_HOLIDAYS:
        hdate = datetime.strptime(date_str, '%Y-%m-%d').date()
        days_away = (hdate - today).days
        if 0 <= days_away <= 60:
            upcoming.append({"holiday": name, "date": date_str, "days_away": days_away})

    upcoming.sort(key=lambda x: x['days_away'])
    next_holiday = upcoming[0] if upcoming else {"holiday": "None in 60 days", "days_away": 61}

    # Demand multiplier: closer to holiday → higher demand
    days_away = next_holiday.get('days_away', 61)
    if days_away <= 7:    holiday_multiplier = 1.35; proximity = "CRITICAL (≤7 days)"
    elif days_away <= 21: holiday_multiplier = 1.20; proximity = "HIGH (≤21 days)"
    elif days_away <= 45: holiday_multiplier = 1.10; proximity = "MODERATE (≤45 days)"
    else:                 holiday_multiplier = 1.00; proximity = "LOW (>45 days)"

    # Combined demand signal
    combined_signal = round(trend_index * holiday_multiplier, 3)
    signal_label = "STRONG BUY WINDOW" if combined_signal > 1.25 else \
                   "BUY WINDOW" if combined_signal > 1.10 else \
                   "NEUTRAL" if combined_signal > 0.95 else "SOFT DEMAND"

    return {
        "status": "ok",
        "category": category,
        "trend_index_vs_baseline": trend_index,
        "trend_source": trend_source,
        "next_holiday": next_holiday,
        "holiday_proximity": proximity,
        "holiday_multiplier": holiday_multiplier,
        "combined_demand_signal": combined_signal,
        "signal_label": signal_label
    }


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 4: simulate_revenue_impact
# ══════════════════════════════════════════════════════════════════════════════
def simulate_revenue_impact(category: str, price_change_pct: float,
                             elasticity_beta: float = None,
                             demand_signal: float = None) -> dict:
    """
    Simulate revenue impact of a price change under optimistic/base/pessimistic scenarios.
    Uses: adjusted_beta = beta × demand_multiplier
          new_qty = current_qty × (1 + price_change_pct)^adjusted_beta
          delta_revenue = new_price × new_qty − current_price × current_qty
    """
    try:
        # Auto-fetch missing inputs
        if elasticity_beta is None:
            elast_result = calculate_price_elasticity(category)
            if elast_result['status'] == 'error':
                return elast_result
            elasticity_beta = elast_result['elasticity_beta']

        if demand_signal is None:
            demand_result = get_demand_signals(category)
            if demand_result['status'] == 'error':
                demand_signal = 1.0  # neutral fallback
            else:
                demand_signal = demand_result['combined_demand_signal']

        # Fetch current avg price and qty from recent data
        sales = query_sales_data(category)
        if sales['status'] == 'error':
            return sales

        df = pd.DataFrame(sales['data'])
        current_price = float(df['avg_price'].median())
        current_qty   = float(df['total_qty'].median())
        current_revenue = current_price * current_qty

        scenarios = {}
        # Base / Optimistic (demand_signal × 1.15) / Pessimistic (demand_signal × 0.85)
        for scenario, ds_mult in [('base', 1.0), ('optimistic', 1.15), ('pessimistic', 0.85)]:
            adj_signal   = demand_signal * ds_mult
            adj_beta     = elasticity_beta * adj_signal
            new_price    = current_price * (1 + price_change_pct)
            new_qty      = current_qty * ((1 + price_change_pct) ** adj_beta)
            new_revenue  = new_price * new_qty
            delta_rev    = new_revenue - current_revenue
            delta_rev_pct = delta_rev / current_revenue if current_revenue > 0 else 0

            scenarios[scenario] = {
                "new_price":       round(new_price, 2),
                "new_qty_index":   round(new_qty / current_qty, 3),
                "qty_change_pct":  round((new_qty - current_qty) / current_qty * 100, 1),
                "revenue_delta":   round(delta_rev, 2),
                "revenue_delta_pct": round(delta_rev_pct * 100, 2),
            }

        action = "RECOMMEND DISCOUNT" if scenarios['base']['revenue_delta_pct'] > 1 and price_change_pct < 0 \
            else "RECOMMEND PRICE INCREASE" if scenarios['base']['revenue_delta_pct'] > 1 and price_change_pct > 0 \
            else "NEUTRAL / MARGINAL IMPACT"

        return {
            "status": "ok",
            "category": category,
            "current_price": round(current_price, 2),
            "current_qty_baseline": round(current_qty, 1),
            "price_change_pct": price_change_pct,
            "elasticity_used": elasticity_beta,
            "demand_signal_used": demand_signal,
            "scenarios": scenarios,
            "recommendation": action
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


print("✅ All 4 tools defined")
print("  Tool 1: query_sales_data")
print("  Tool 2: calculate_price_elasticity")
print("  Tool 3: get_demand_signals")
print("  Tool 4: simulate_revenue_impact")


✅ All 4 tools defined
  Tool 1: query_sales_data
  Tool 2: calculate_price_elasticity
  Tool 3: get_demand_signals
  Tool 4: simulate_revenue_impact


### 3.1 Quick Tool Smoke Test (no LLM needed)

In [17]:
# Sanity check all 4 tools before wiring the LLM
test_category = 'esporte_lazer'

print("── Tool 1: query_sales_data ──")
r1 = query_sales_data(test_category)
print(f"  Status: {r1['status']}, buckets: {r1.get('n_price_buckets')}, total orders: {r1.get('total_orders')}")

print("\n── Tool 2: calculate_price_elasticity ──")
r2 = calculate_price_elasticity(test_category)
print(f"  β = {r2.get('elasticity_beta')}, R² = {r2.get('r_squared')}, p = {r2.get('p_value')}")
print(f"  Interpretation: {r2.get('interpretation')} | {r2.get('rule_of_thumb')}")

print("\n── Tool 3: get_demand_signals ──")
r3 = get_demand_signals(test_category)
print(f"  Trend index: {r3.get('trend_index_vs_baseline')} ({r3.get('trend_source')[:40]})")
print(f"  Next holiday: {r3.get('next_holiday')}")
print(f"  Signal: {r3.get('signal_label')}")

print("\n── Tool 4: simulate_revenue_impact ──")
r4 = simulate_revenue_impact(test_category, price_change_pct=-0.10)
print(f"  Base scenario: {r4['scenarios']['base']['revenue_delta_pct']:+.1f}% revenue")
print(f"  Recommendation: {r4.get('recommendation')}")


── Tool 1: query_sales_data ──
  Status: ok, buckets: 14, total orders: 1360738

── Tool 2: calculate_price_elasticity ──
  β = -4.141, R² = 0.84, p = 0.0
  Interpretation: elastic | A 10% price change → ~41.4% volume change

── Tool 3: get_demand_signals ──
  Trend index: 1.0 (seasonal_estimate (pytrends unavailable:)
  Next holiday: {'holiday': 'None in 60 days', 'days_away': 61}
  Signal: NEUTRAL

── Tool 4: simulate_revenue_impact ──
  Base scenario: +99.1% revenue
  Recommendation: RECOMMEND DISCOUNT


## 4. Claude Tool Definitions (JSON Schema)

In [18]:
TOOLS = [
    {
        "name": "query_sales_data",
        "description": "Query historical Olist e-commerce order data for a product category. Returns price-volume distribution by price bucket.",
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "description": "Olist product category name (Portuguese), e.g. 'esporte_lazer', 'eletronicos'"},
                "start_date": {"type": "string", "description": "Start year-month, format YYYY-MM (default '2017-01')"},
                "end_date":   {"type": "string", "description": "End year-month, format YYYY-MM (default '2018-08')"}
            },
            "required": ["category"]
        }
    },
    {
        "name": "calculate_price_elasticity",
        "description": "Calculate price elasticity of demand using log-linear regression (ln Q = α + β ln P). Returns elasticity coefficient β, R², confidence interval.",
        "input_schema": {
            "type": "object",
            "properties": {
                "category":   {"type": "string", "description": "Olist product category name"},
                "start_date": {"type": "string", "description": "Start year-month YYYY-MM"},
                "end_date":   {"type": "string", "description": "End year-month YYYY-MM"}
            },
            "required": ["category"]
        }
    },
    {
        "name": "get_demand_signals",
        "description": "Fetch real-time demand signals: Google Trends search index and Brazilian holiday proximity. Returns combined demand multiplier and signal label.",
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "description": "Product category (English or Portuguese keywords)"},
                "country":  {"type": "string", "description": "Country code for Google Trends (default 'BR' for Brazil)"}
            },
            "required": ["category"]
        }
    },
    {
        "name": "simulate_revenue_impact",
        "description": "Simulate revenue impact of a proposed price change under base/optimistic/pessimistic scenarios using price elasticity and demand signal.",
        "input_schema": {
            "type": "object",
            "properties": {
                "category":        {"type": "string", "description": "Product category"},
                "price_change_pct":{"type": "number", "description": "Proposed price change as decimal (e.g. -0.10 for -10%, +0.05 for +5%)"},
                "elasticity_beta": {"type": "number", "description": "Optional: pre-computed elasticity β (auto-fetched if omitted)"},
                "demand_signal":   {"type": "number", "description": "Optional: pre-computed demand multiplier (auto-fetched if omitted)"}
            },
            "required": ["category", "price_change_pct"]
        }
    }
]

# Python dispatch map
TOOL_FUNCTIONS = {
    "query_sales_data":         query_sales_data,
    "calculate_price_elasticity": calculate_price_elasticity,
    "get_demand_signals":       get_demand_signals,
    "simulate_revenue_impact":  simulate_revenue_impact,
}

print(f"✅ {len(TOOLS)} tool schemas registered for Claude")


✅ 4 tool schemas registered for Claude


## 5. Planner Agent

In [19]:
# ── Few-shot prompt with XML tagging (required by course) ────────────────────
PLANNER_SYSTEM = """
<context>
You are the Planner agent for PriceIQ, an e-commerce dynamic pricing intelligence system
operating on the Olist Brazilian e-commerce dataset (2016–2018, 100K orders).

Valid Olist category names (Portuguese):
esporte_lazer | eletronicos | moda_bolsas | casa_mesa_banho | informatica_acessorios

Your ONLY job is to parse the user query and output a structured JSON execution plan.
Do NOT call tools yourself. Do NOT answer the pricing question. Output ONLY a JSON object.
</context>

<task>
Given a user query about product pricing in Brazil, return a JSON plan with these fields:
- category: Olist category name (Portuguese)
- price_change_pct: the proposed price change as a decimal (e.g. -0.10)
- tools_needed: ordered list of tools to call
- reasoning: 1-sentence explanation of why this sequence
</task>

<rules>
- If user says "discount" or "reduce" → price_change_pct is negative
- If user says "raise", "increase", "premium" → price_change_pct is positive
- If price magnitude not specified → default to -0.10 (discount) or +0.10 (increase)
- Always include all 4 tools in sequence: query_sales_data → calculate_price_elasticity → get_demand_signals → simulate_revenue_impact
- Map English category names to Portuguese (e.g. sports → esporte_lazer, electronics → eletronicos)
</rules>

<examples>
Query: "Should we discount sports equipment by 15% before Black Friday?"
Plan:
{
  "category": "esporte_lazer",
  "price_change_pct": -0.15,
  "tools_needed": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"],
  "reasoning": "User wants 15% discount on sports; need elasticity to quantify volume response and demand signal to confirm holiday timing."
}

Query: "Is it a good idea to raise electronics prices by 5%?"
Plan:
{
  "category": "eletronicos",
  "price_change_pct": 0.05,
  "tools_needed": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"],
  "reasoning": "User wants to raise prices; need elasticity to check if inelastic enough to absorb increase and demand signal to confirm market conditions."
}

Query: "What happens to revenue if we cut fashion bag prices?"
Plan:
{
  "category": "moda_bolsas",
  "price_change_pct": -0.10,
  "tools_needed": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"],
  "reasoning": "User asks about cutting prices; defaulting to -10% and running full analysis to quantify revenue impact."
}
</examples>
"""

def run_planner(user_query: str) -> dict:
    """
    Planner agent: parse user intent → return structured execution plan.
    Uses claude-sonnet for complex NL understanding.
    """
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        system=PLANNER_SYSTEM,
        messages=[{"role": "user", "content": user_query}]
    )

    raw = response.content[0].text.strip()

    # Extract JSON from response (may be wrapped in markdown code block)
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    try:
        plan = json.loads(raw)
        print(f"🧠 Planner → category: {plan['category']}, Δprice: {plan['price_change_pct']:+.0%}")
        print(f"   Reasoning: {plan['reasoning']}")
        return plan
    except json.JSONDecodeError:
        # Fallback plan if parsing fails
        print(f"⚠️  Planner JSON parse failed, using fallback plan")
        return {
            "category": "esporte_lazer",
            "price_change_pct": -0.10,
            "tools_needed": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"],
            "reasoning": "Fallback plan: default category and price change"
        }

print("✅ Planner agent defined")


✅ Planner agent defined


## 6. Executor Agent (Manual Tool-Use Loop)

In [20]:
# ── Memory management ────────────────────────────────────────────────────────
def summarize_tool_results(tool_results: list) -> str:
    """
    Condense tool output history into a compact summary for memory management.
    Prevents context window overflow in multi-turn scenarios.
    """
    summary_parts = []
    for r in tool_results:
        name = r['tool_name']
        result = r['result']
        if name == 'query_sales_data' and result.get('status') == 'ok':
            summary_parts.append(
                f"[query_sales_data] {result['category']}: {result['n_price_buckets']} price buckets, "
                f"price range ${result['price_range']['min']}–${result['price_range']['max']}, "
                f"{result['total_orders']} total orders"
            )
        elif name == 'calculate_price_elasticity' and result.get('status') == 'ok':
            summary_parts.append(
                f"[calculate_price_elasticity] β={result['elasticity_beta']}, R²={result['r_squared']}, "
                f"p={result['p_value']}, {result['interpretation']}, CI=[{result['confidence_interval_95'][0]}, {result['confidence_interval_95'][1]}]"
            )
        elif name == 'get_demand_signals' and result.get('status') == 'ok':
            summary_parts.append(
                f"[get_demand_signals] trend_index={result['trend_index_vs_baseline']}, "
                f"next_holiday={result['next_holiday']}, signal={result['signal_label']}, "
                f"combined={result['combined_demand_signal']}"
            )
        elif name == 'simulate_revenue_impact' and result.get('status') == 'ok':
            s = result['scenarios']
            summary_parts.append(
                f"[simulate_revenue_impact] base={s['base']['revenue_delta_pct']:+.1f}%, "
                f"optimistic={s['optimistic']['revenue_delta_pct']:+.1f}%, "
                f"pessimistic={s['pessimistic']['revenue_delta_pct']:+.1f}%, "
                f"recommendation={result['recommendation']}"
            )
        else:
            summary_parts.append(f"[{name}] status={result.get('status')}: {result.get('message', 'ok')}")
    return " | ".join(summary_parts)


def run_executor(user_query: str, plan: dict, verbose: bool = True) -> dict:
    """
    Executor agent: manual tool_use loop using Claude SDK.
    Implements kill switch (MAX_ITERATIONS), memory management, and graceful degradation.
    """
    MAX_ITERATIONS = 8  # Kill switch: prevent infinite loops

    # ── Build initial context with memory summary ────────────────────────────
    category     = plan['category']
    price_change = plan['price_change_pct']
    tools_needed = plan.get('tools_needed', list(TOOL_FUNCTIONS.keys()))

    system_prompt = f"""You are the Executor agent for PriceIQ.

Your task is to call the tools listed below IN SEQUENCE to answer this pricing query.
The Planner has determined:
- Category: {category}
- Proposed price change: {price_change:+.0%}
- Tools to call: {' → '.join(tools_needed)}

Call each tool with the correct arguments. After all tools have returned results,
synthesize a FINAL ANSWER with:
1. Elasticity interpretation (elastic/inelastic)
2. Demand signal assessment
3. Revenue projections (base / optimistic / pessimistic)
4. Clear BUY / NO-BUY recommendation with rationale

Be concise and data-driven. Do not speculate beyond the tool outputs."""

    messages = [{"role": "user", "content": user_query}]
    tool_results_log = []
    iteration = 0

    if verbose:
        print(f"\n{'═'*60}")
        print(f"  EXECUTOR AGENT — {category} | {price_change:+.0%} price change")
        print(f"{'═'*60}")

    while iteration < MAX_ITERATIONS:
        iteration += 1

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",  # Cost-efficient executor model
            max_tokens=1024,
            system=system_prompt,
            tools=TOOLS,
            messages=messages
        )

        if verbose:
            print(f"\n[Iter {iteration}] stop_reason={response.stop_reason}")

        # ── End condition: no more tool calls ────────────────────────────────
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, 'text'):
                    final_text += block.text
            if verbose:
                print(f"\n{'─'*60}")
                print("FINAL ANSWER:")
                print(final_text)
                print(f"{'─'*60}")
                print(f"\nTotal iterations: {iteration} | Tools called: {len(tool_results_log)}")
            return {
                "status": "complete",
                "final_answer": final_text,
                "tool_results": tool_results_log,
                "iterations": iteration
            }

        # ── Process tool_use blocks ──────────────────────────────────────────
        if response.stop_reason == "tool_use":
            # Add assistant's response to message history
            messages.append({"role": "assistant", "content": response.content})

            tool_result_blocks = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_name = block.name
                tool_input = block.input

                if verbose:
                    print(f"  → ACTION: {tool_name}({json.dumps(tool_input, ensure_ascii=False)})")

                # ── Dispatch tool call ───────────────────────────────────────
                if tool_name in TOOL_FUNCTIONS:
                    try:
                        result = TOOL_FUNCTIONS[tool_name](**tool_input)
                    except Exception as e:
                        result = {"status": "error", "message": f"Tool execution error: {str(e)}"}
                else:
                    result = {"status": "error", "message": f"Unknown tool: {tool_name}"}

                if verbose:
                    status = result.get('status', 'unknown')
                    if status == 'ok':
                        # Print key result snippet
                        if tool_name == 'calculate_price_elasticity':
                            print(f"  ← OBSERVATION: β={result.get('elasticity_beta')}, R²={result.get('r_squared')}, {result.get('interpretation')}")
                        elif tool_name == 'get_demand_signals':
                            print(f"  ← OBSERVATION: trend={result.get('trend_index_vs_baseline')}, signal={result.get('signal_label')}")
                        elif tool_name == 'simulate_revenue_impact':
                            base = result['scenarios']['base']['revenue_delta_pct']
                            print(f"  ← OBSERVATION: base revenue Δ={base:+.1f}%, rec={result.get('recommendation')}")
                        else:
                            print(f"  ← OBSERVATION: status=ok, {result.get('n_price_buckets', '')} buckets")
                    else:
                        print(f"  ← ERROR: {result.get('message')}")

                tool_results_log.append({"tool_name": tool_name, "result": result})

                tool_result_blocks.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result)
                })

            # Add tool results to messages
            messages.append({"role": "user", "content": tool_result_blocks})

            # ── Memory management: summarize if history growing large ────────
            total_msg_chars = sum(len(str(m)) for m in messages)
            if total_msg_chars > 8000 and len(tool_results_log) >= 2:
                memory_summary = summarize_tool_results(tool_results_log)
                # Compress old tool results in messages (keep last 2 turns)
                system_prompt += f"\n\n<memory_summary>Previous tool results: {memory_summary}</memory_summary>"
                messages = messages[-4:]  # Keep only last 4 messages
                if verbose:
                    print(f"  [Memory] History compressed to {len(messages)} messages")

    # ── Kill switch triggered ────────────────────────────────────────────────
    if verbose:
        print(f"⚠️  Kill switch: reached MAX_ITERATIONS={MAX_ITERATIONS}")
    return {
        "status": "partial",
        "final_answer": "Analysis incomplete: maximum iterations reached. Partial results available.",
        "tool_results": tool_results_log,
        "iterations": iteration
    }

print("✅ Executor agent defined (manual tool_use loop)")
print(f"   Kill switch: MAX_ITERATIONS=8")
print(f"   Memory: summarization triggered at >8K chars")
print(f"   Graceful degradation: all tools have try/except + fallback")


✅ Executor agent defined (manual tool_use loop)
   Kill switch: MAX_ITERATIONS=8
   Memory: summarization triggered at >8K chars
   Graceful degradation: all tools have try/except + fallback


## 7. PriceIQ Agent — Main Entry Point

In [21]:
def priceiq_agent(user_query: str, verbose: bool = True) -> dict:
    """
    Main PriceIQ agent: Planner → Executor pipeline.

    Args:
        user_query: natural language pricing question
        verbose: print step-by-step trace (default True)

    Returns:
        dict with final_answer, tool_results, and metadata
    """
    if verbose:
        print(f"\n{'━'*60}")
        print(f"  PRICEIQ AGENT")
        print(f"  Query: {user_query}")
        print(f"{'━'*60}")
        print("\n[PLANNER]")

    # Step 1: Planner parses intent
    plan = run_planner(user_query)

    if verbose:
        print(f"\n[EXECUTOR]")

    # Step 2: Executor runs tools
    result = run_executor(user_query, plan, verbose=verbose)
    result['plan'] = plan
    result['query'] = user_query

    return result

print("✅ priceiq_agent() ready")


✅ priceiq_agent() ready


## 8. Demo — Run the Agent

In [22]:
# ── Primary demo query ───────────────────────────────────────────────────────
DEMO_QUERY = "Should we discount sports equipment in Brazil this November?"

result = priceiq_agent(DEMO_QUERY, verbose=True)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PRICEIQ AGENT
  Query: Should we discount sports equipment in Brazil this November?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[PLANNER]
🧠 Planner → category: esporte_lazer, Δprice: -10%
   Reasoning: User asks about discounting sports equipment; defaulting to -10% and running full analysis to evaluate November seasonality and revenue impact.

[EXECUTOR]

════════════════════════════════════════════════════════════
  EXECUTOR AGENT — esporte_lazer | -10% price change
════════════════════════════════════════════════════════════

[Iter 1] stop_reason=tool_use
  → ACTION: query_sales_data({"category": "esporte_lazer"})
  ← OBSERVATION: status=ok, 14 buckets

[Iter 2] stop_reason=tool_use
  → ACTION: calculate_price_elasticity({"category": "esporte_lazer"})
  ← OBSERVATION: β=-4.141, R²=0.84, elastic
  → ACTION: get_demand_signals({"category": "esporte_lazer", "country": "BR"})
  ← OBSERVATION: trend=1.82, s

## 9. Additional Test Cases (5 Seed Cases)

In [23]:
# Gold standard test cases (run each and inspect output)
TEST_CASES = [
    # 1. Happy path
    ("Should we discount sports equipment by 10% before Black Friday?",
     "Happy Path: elastic category + high demand window"),
    # 2. Edge case: inelastic category
    ("Is a 10% price increase for electronics viable right now?",
     "Edge Case: test inelastic behavior, expect low volume impact"),
    # 3. Complex: fashion
    ("What happens to fashion bag revenue if we cut prices 15%?",
     "Complex 1: moderate elasticity, check seasonal demand"),
    # 4. Ambiguous: no price specified
    ("Should we run a promotion on home goods?",
     "Complex 2: ambiguous — Planner must infer default discount"),
    # 5. Adversarial: category mismatch
    ("Should we raise prices on 'tecnologia'?",
     "Adversarial: non-standard category name — Planner must map to valid Olist name"),
]

print("Available test cases:")
for i, (query, label) in enumerate(TEST_CASES, 1):
    print(f"  [{i}] {label}")
    print(f"      Query: {query}")
    print()

print("To run a test case:")
print("  result = priceiq_agent(TEST_CASES[0][0], verbose=True)")


Available test cases:
  [1] Happy Path: elastic category + high demand window
      Query: Should we discount sports equipment by 10% before Black Friday?

  [2] Edge Case: test inelastic behavior, expect low volume impact
      Query: Is a 10% price increase for electronics viable right now?

  [3] Complex 1: moderate elasticity, check seasonal demand
      Query: What happens to fashion bag revenue if we cut prices 15%?

  [4] Complex 2: ambiguous — Planner must infer default discount
      Query: Should we run a promotion on home goods?

  [5] Adversarial: non-standard category name — Planner must map to valid Olist name
      Query: Should we raise prices on 'tecnologia'?

To run a test case:
  result = priceiq_agent(TEST_CASES[0][0], verbose=True)


### 9.1 Run a Single Test Case

In [24]:
# Change index 0–4 to run different test cases
TEST_IDX = 1  # Edge case: electronics price increase

query, label = TEST_CASES[TEST_IDX]
print(f"Running: {label}\n")
result2 = priceiq_agent(query, verbose=True)


Running: Edge Case: test inelastic behavior, expect low volume impact


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PRICEIQ AGENT
  Query: Is a 10% price increase for electronics viable right now?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[PLANNER]
🧠 Planner → category: eletronicos, Δprice: +10%
   Reasoning: User wants 10% increase on electronics; need elasticity to check price sensitivity and demand signals to validate current market conditions support increase.

[EXECUTOR]

════════════════════════════════════════════════════════════
  EXECUTOR AGENT — eletronicos | +10% price change
════════════════════════════════════════════════════════════

[Iter 1] stop_reason=tool_use
  → ACTION: query_sales_data({"category": "eletronicos"})
  ← OBSERVATION: status=ok, 36 buckets
  → ACTION: calculate_price_elasticity({"category": "eletronicos"})
  ← OBSERVATION: β=-3.381, R²=0.786, elastic
  → ACTION: get_demand_signals({"category": "eletronicos", "count

## 10. Results Summary

In [25]:
def print_summary(result: dict):
    """Print a clean summary of the agent's output."""
    print("\n" + "╔" + "═"*58 + "╗")
    print("║  PRICEIQ AGENT — RESULTS SUMMARY" + " "*24 + "║")
    print("╠" + "═"*58 + "╣")
    print(f"║  Query: {result['query'][:50]:<50} ║")
    plan = result.get('plan', {})
    print(f"║  Category: {plan.get('category', 'N/A'):<48} ║")
    print(f"║  Price Change: {plan.get('price_change_pct', 0):+.0%}{'':<44} ║")
    print(f"║  Status: {result['status']:<50} ║")
    print(f"║  Iterations: {result['iterations']:<46} ║")
    print(f"║  Tools Called: {len(result['tool_results']):<44} ║")
    print("╠" + "═"*58 + "╣")

    # Find simulation result
    sim = next((r['result'] for r in result['tool_results']
                if r['tool_name'] == 'simulate_revenue_impact' and r['result'].get('status') == 'ok'), None)
    if sim:
        s = sim['scenarios']
        print(f"║  Revenue Impact (Base):       {s['base']['revenue_delta_pct']:+6.1f}%{'':<26} ║")
        print(f"║  Revenue Impact (Optimistic): {s['optimistic']['revenue_delta_pct']:+6.1f}%{'':<26} ║")
        print(f"║  Revenue Impact (Pessimistic):{s['pessimistic']['revenue_delta_pct']:+6.1f}%{'':<26} ║")
        print(f"║  Recommendation: {sim['recommendation']:<41} ║")
    print("╚" + "═"*58 + "╝")

print_summary(result)



╔══════════════════════════════════════════════════════════╗
║  PRICEIQ AGENT — RESULTS SUMMARY                        ║
╠══════════════════════════════════════════════════════════╣
║  Query: Should we discount sports equipment in Brazil this ║
║  Category: esporte_lazer                                    ║
║  Price Change: -10%                                             ║
║  Status: complete                                           ║
║  Iterations: 4                                              ║
║  Tools Called: 4                                            ║
╠══════════════════════════════════════════════════════════╣
║  Revenue Impact (Base):        +99.1%                           ║
║  Revenue Impact (Optimistic): +124.3%                           ║
║  Revenue Impact (Pessimistic): +76.8%                           ║
║  Recommendation: RECOMMEND DISCOUNT                        ║
╚══════════════════════════════════════════════════════════╝


---
## ✅ Phase 1 Prototype — Complete

**What this notebook demonstrates:**
- Planner + Executor dual-agent architecture (Track B: Claude Agent SDK)
- Manual `tool_use` loop (no managed agents)
- 4 working tools with real statistical computation (scipy log-linear regression)
- Few-shot prompting with XML tagging (`<context>`, `<task>`, `<examples>`, `<rules>`, `<memory_summary>`)
- Manual memory management (history summarization at 8K chars)
- Graceful degradation (pytrends fallback → seasonal estimates)
- Kill switch (MAX_ITERATIONS = 8)
- No hard-coded test answers
- No API key in code (uses Colab Secrets)

**Next steps (Phase 2):**
- Load real Olist SQLite data from Kaggle
- Expand to all 73 Olist categories
- Add OpenWeather API integration for weather-sensitive categories
- Implement 10-case gold standard evaluation suite
- Red-team testing and failure log
